In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and indices
n = 5  # Number of points
K = 3  # Number of products

np.random.seed(0)

# Distance cost matrix
C_t = 3 * np.random.rand(n, n)

# Initial shelf life, required shelf life, and quality reduction factors
SL_0 = 700 * np.random.rand(K)  # Initial shelf life
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor

# Priority customers
P = np.random.choice([0, 1], size=n, p=[.7, .3])
alpha = 7

# Nominal values for R and T (means of the distributions)
R_nom = 2 * np.random.rand(n)  # Nominal delay at each stop
T_nom = 5 * np.random.rand(n, n)  # Nominal travel time from point i to j

# Scenario generation for uncertain parameters (R and T)
n_scenarios = 10  # Number of scenarios for DRO approximation
R_scenarios = [R_nom + 0.5 * np.random.randn(n) for _ in range(n_scenarios)]  # Random deviations for R
T_scenarios = [T_nom + 0.5 * np.random.randn(n, n) for _ in range(n_scenarios)]  # Random deviations for T

In [ ]:
# Create model
model = gp.Model('DRO Perishable Produce Transportation')

In [ ]:
# Variables
x = model.addVars(n, n, vtype=GRB.BINARY)  # 1 if arc i-j is part of the route
u = model.addVars(n, lb=0, ub=float('inf'), vtype=GRB.CONTINUOUS)  # Auxiliary variable for subtour elimination

In [ ]:
# Constraints

# Each node is entered once
model.addConstrs((gp.quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), name='const1')

# No self-loops
model.addConstrs((x[i, j] == 0 for i in range(n) for j in range(n) if i == j), name='const2')

# Each node is exited once
model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), name='const3')

# MTZ Constraints (Subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n for i in range(n) for j in range(n) if i != j), name='const4')

# DRO in-transit time constraint over worst-case scenarios
for k in range(K):
    for s in range(n_scenarios):
        model.addConstr(
            gp.quicksum((R_scenarios[s][i] + T_scenarios[s][i, j]) * x[i, j] for i in range(n) for j in range(n))
            <= Q[k] * SL_0[k] - SL_r[k], name=f'dro_time_constraint_k_{k}_s_{s}'
        )

In [ ]:
# Objective function: Minimize cost minus priority term
cost_term = gp.quicksum(C_t[i, j] * x[i, j] for i in range(n) for j in range(n))
priority_term = gp.quicksum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

# Set the objective (minimize cost minus priority term)
model.setObjective(cost_term - priority_term, GRB.MINIMIZE)

In [ ]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (win64)
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads
Optimize a model with 65 rows, 30 columns and 865 nonzeros
Model fingerprint: 0xf7df2cca
Variable types: 5 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [2e-01, 7e+00]
  Objective range  [6e-02, 1e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+02]
Found heuristic solution: objective -2.4697255
Presolve removed 35 rows and 5 columns
Presolve time: 0.00s
Presolved: 30 rows, 25 columns, 100 nonzeros
Variable types: 5 continuous, 20 integer (20 binary)

Root relaxation: objective -8.769467e+00, 8 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0      -8.7694667   -8.76947  0.00%     -    0s

Explored 0 nodes (8 simplex iterations) in 0.04 seconds
Thread c

In [ ]:
# Display results
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model.objVal}")

    for var in model.getVars():
        if var.X > 0:
            print(f"{var.VarName}: {var.X}")
else:
    print(f"Optimization was unsuccessful. Status code: {model.status}")

Optimal objective value: -8.769466658135908
C3: 1.0
C5: 1.0
C14: 1.0
C16: 1.0
C22: 1.0
